# Linear Regression — companion notebook

Runnable companion for [`linear-regression.md`](./linear-regression.md).

1. Fit a synthetic regression via normal equations, sklearn, and gradient descent — confirm they agree.
2. Visualize the fit.
3. Break the normal equations with collinear features; show the SVD pseudoinverse rescues the minimum-norm solution.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

## 1. Synthetic data, three ways to fit

True model: $y = 2 + 3 x_1 - 1.5 x_2 + \varepsilon$, $\varepsilon \sim \mathcal{N}(0, 0.5^2)$.

In [ ]:
n, p = 200, 2
X_raw = rng.standard_normal((n, p))
beta_true = np.array([3.0, -1.5])
intercept_true = 2.0
y = intercept_true + X_raw @ beta_true + 0.5 * rng.standard_normal(n)

# Augment with a column of ones so beta[0] is the intercept.
X = np.hstack([np.ones((n, 1)), X_raw])

# (a) Normal equations via a linear solve (never invert explicitly).
beta_normal = np.linalg.solve(X.T @ X, X.T @ y)

# (b) sklearn (it fits the intercept separately).
reg = LinearRegression().fit(X_raw, y)
beta_sklearn = np.concatenate([[reg.intercept_], reg.coef_])

# (c) Gradient descent on (1/(2n)) ||y - X beta||^2
def gradient_descent(X, y, eta=0.05, steps=2000):
    beta = np.zeros(X.shape[1])
    n = len(y)
    losses = []
    for _ in range(steps):
        grad = X.T @ (X @ beta - y) / n
        beta -= eta * grad
        losses.append(0.5 * np.mean((y - X @ beta) ** 2))
    return beta, losses

beta_gd, losses = gradient_descent(X, y)

print('true             :', np.concatenate([[intercept_true], beta_true]))
print('normal equations :', beta_normal)
print('sklearn          :', beta_sklearn)
print('gradient descent :', beta_gd)
print('max disagreement :', np.max(np.abs(beta_normal - beta_gd)))

In [ ]:
# Visualize: project onto x1 (1D slice), show fit vs. data.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

order = np.argsort(X_raw[:, 0])
axes[0].scatter(X_raw[:, 0], y, s=12, alpha=0.5, label='data')
# At fixed x2 = mean(x2), plot the fit line.
x2_mean = X_raw[:, 1].mean()
x1_grid = np.linspace(X_raw[:, 0].min(), X_raw[:, 0].max(), 100)
y_hat_line = beta_normal[0] + beta_normal[1] * x1_grid + beta_normal[2] * x2_mean
axes[0].plot(x1_grid, y_hat_line, 'tab:red', lw=2, label='OLS fit at x2=mean')
axes[0].set_xlabel('x1'); axes[0].set_ylabel('y'); axes[0].legend(); axes[0].set_title('Fit slice')

axes[1].semilogy(losses)
axes[1].set_xlabel('step'); axes[1].set_ylabel('MSE (log)')
axes[1].set_title('Gradient descent convergence to OLS minimum')
axes[1].axhline(0.5 * np.mean((y - X @ beta_normal) ** 2), color='tab:red', ls='--', label='OLS optimum')
axes[1].legend()
plt.tight_layout(); plt.show()

## 2. Collinearity breaks the normal equations

Append a column equal to $x_1$ exactly. Now $X^\top X$ is singular — `np.linalg.solve` fails. SVD pseudoinverse returns the minimum-norm solution among the infinitely many minimizers.

In [ ]:
# Duplicate x1 as x3 — perfectly collinear features.
X_bad = np.hstack([X, X_raw[:, [0]]])  # columns: [1, x1, x2, x1]
rank = np.linalg.matrix_rank(X_bad)
print(f'X_bad shape = {X_bad.shape}, rank = {rank}  (deficient)')

G = X_bad.T @ X_bad
print(f'cond(X^T X)       = {np.linalg.cond(G):.3e}')

try:
    np.linalg.solve(G, X_bad.T @ y)
except np.linalg.LinAlgError as e:
    print('normal-equations solve raised:', e)

In [ ]:
# (a) SVD pseudoinverse -> minimum-norm least-squares solution.
beta_pinv = np.linalg.pinv(X_bad) @ y

# (b) Ridge regression (any lambda > 0 stabilizes the inversion).
lam = 1e-3
beta_ridge = np.linalg.solve(X_bad.T @ X_bad + lam * np.eye(X_bad.shape[1]), X_bad.T @ y)

# Both yield identical predictions even though the coefficients differ.
yhat_pinv = X_bad @ beta_pinv
yhat_ridge = X_bad @ beta_ridge
yhat_full = X @ beta_normal

print('beta (pinv) :', beta_pinv)
print('beta (ridge):', beta_ridge)
print(f'||y_hat_pinv - y_hat_full||  = {np.linalg.norm(yhat_pinv - yhat_full):.2e}')
print(f'||y_hat_ridge - y_hat_full|| = {np.linalg.norm(yhat_ridge - yhat_full):.2e}')
print('Note: pinv split the x1 coefficient evenly across the two collinear columns.')